### Machine Learning | Assignment 3 - Evolutionary Algorithms

145873 | Sezer Tetik

#### Exercise 1

In [ ]:
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

**a)** Function that produces a random bit pattern of a given size, coded as a string of `0`s and `1`s.

`rng.getrandbits(size)` returns an integer drawn uniformly from $[0, 2^{size})$, i.e. `size` independent fair bits, so every one of the $2^{size}$ possible patterns is equally likely. Formatting it with `f'0{size}b'` turns it into a binary string and left-pads with zeros, so leading `0` bits are not lost.

The random generator is passed as an argument (`random.Random(seed)`) so each experiment owns its own seeded generator and results are reproducible without depending on global state.

In [ ]:
def generate_random_pattern(size: int, rng: random.Random) -> str:
    if size < 1:
        raise ValueError("size must be >= 1")

    return format(rng.getrandbits(size), f"0{size}b")

In [ ]:
rng = random.Random(42)

for size in [2, 4, 8, 12, 16]:
    pattern = generate_random_pattern(size, rng)
    print(f"{size:>2} bits -> {pattern}")

Sanity checks: correct length, only `0`/`1`, same seed gives same pattern, and each bit is close to 50% ones over many draws (no bias).

In [ ]:
size = 16

assert generate_random_pattern(size, random.Random(1)) == generate_random_pattern(size, random.Random(1))

rng = random.Random(0)
patterns = [generate_random_pattern(size, rng) for _ in range(100_000)]

assert all(len(p) == size and set(p) <= {"0", "1"} for p in patterns)

ones_ratio_per_bit = [sum(p[i] == "1" for p in patterns) / len(patterns) for i in range(size)]
[round(r, 3) for r in ones_ratio_per_bit]

**b)** Brute-force search: keep generating random patterns with `generate_random_pattern` until one equals the target, counting the attempts and measuring the elapsed time with `time.perf_counter()`.

Each guess is independent and matches the target with probability $p = 1/2^n$, so the number of attempts follows a geometric distribution with $E[\text{attempts}] = 2^n$ and standard deviation $\sqrt{1-p}/p \approx 2^n$. That $2^n$ is the theoretical baseline the measurements are compared against: every 4 extra bits should multiply the attempts, and therefore the time, by $2^4 = 16$.

Guesses are drawn with replacement (the same wrong pattern can be tried again). A systematic enumeration without repetition would need at most $2^n$ and on average $(2^n+1)/2$ attempts, so random brute force costs about twice as much on average.

In [ ]:
def brute_force(target: str, rng: random.Random) -> tuple[int, float]:
    size = len(target)
    attempts = 0
    start = time.perf_counter()

    while True:
        attempts += 1

        if generate_random_pattern(size, rng) == target:
            return attempts, time.perf_counter() - start

**Experiment setup**

- Each trial uses its own `random.Random(seed)` with seeds $0 \dots 29$. The first draw is the target, the following draws are the guesses. Same seed means same target and same guesses, so the attempt counts are exactly reproducible; only the times vary with machine load.
- The next exercises reuse these seeds, so every method searches for the same 30 targets, which keeps the comparison fair.
- Sizes go 2, 4, 8, 12, 16, … in steps of 4. The 30 trials of one size share a 15-minute budget, checked after every trial. Once a size goes over the budget, no larger size is tried. Only sizes with all 30 trials go into the plots.

In [ ]:
SEEDS = list(range(30))
SIZES = [2, 4, 8, 12, 16, 20, 24, 28, 32]
TIME_BUDGET = 15 * 60  # seconds, for all 30 trials of one size

results = {}  # size -> {"attempts": array, "times": array}

for size in SIZES:
    attempts, times = [], []
    size_start = time.perf_counter()

    for seed in SEEDS:
        rng = random.Random(seed)
        target = generate_random_pattern(size, rng)
        n_attempts, elapsed = brute_force(target, rng)

        attempts.append(n_attempts)
        times.append(elapsed)

        if time.perf_counter() - size_start > TIME_BUDGET:
            break

    total = time.perf_counter() - size_start
    print(f"{size:>2} bits | {len(attempts):>2}/{len(SEEDS)} trials | {total:8.2f} s")

    if len(attempts) == len(SEEDS):
        results[size] = {"attempts": np.array(attempts), "times": np.array(times)}

    if total > TIME_BUDGET:
        print(f"{size} bits went over the {TIME_BUDGET / 60:.0f} min budget -> stopping")
        break

In [ ]:
BOX_COLOR = "#2a78d6"
BOX_FILL = "#cde2fb"
BASELINE_COLOR = "#eb6834"
MUTED = "#898781"
GRID = "#e1e0d9"

sizes = list(results)
positions = list(range(len(sizes)))


def boxplot_vs_bits(data, title, ylabel, log_base=10, yticks=None, baseline=None, baseline_label=None):
    fig, ax = plt.subplots(figsize=(9, 5))

    ax.boxplot(
        data,
        positions=positions,
        widths=0.5,
        patch_artist=True,
        showmeans=True,
        boxprops={"facecolor": BOX_FILL, "edgecolor": BOX_COLOR, "linewidth": 1},
        medianprops={"color": BOX_COLOR, "linewidth": 2},
        whiskerprops={"color": BOX_COLOR, "linewidth": 1},
        capprops={"color": BOX_COLOR, "linewidth": 1},
        flierprops={"marker": "o", "markersize": 4, "markerfacecolor": "none", "markeredgecolor": MUTED},
        meanprops={"marker": "D", "markersize": 6, "markerfacecolor": "white", "markeredgecolor": BOX_COLOR},
    )

    handles = [
        Patch(facecolor=BOX_FILL, edgecolor=BOX_COLOR, label=f"{len(SEEDS)} trials: box = IQR, line = median, whiskers = 1.5 IQR"),
        Line2D([], [], marker="D", linestyle="none", markerfacecolor="white", markeredgecolor=BOX_COLOR, label="Mean"),
    ]

    if baseline is not None:
        ax.plot(positions, baseline, color=BASELINE_COLOR, linewidth=2, marker="o", markersize=4, zorder=1)
        handles.append(Line2D([], [], color=BASELINE_COLOR, linewidth=2, label=baseline_label))

    ax.set_yscale("log", base=log_base)
    if yticks is not None:
        ax.set_yticks(yticks)
    ax.set_xticks(positions, sizes)
    ax.set_xlabel("Pattern size (bits)")
    ax.set_ylabel(ylabel)
    ax.set_title(title, loc="left")

    ax.grid(axis="y", which="major", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color(MUTED)
    ax.tick_params(colors=MUTED, labelcolor="#52514e")
    ax.legend(handles=handles, frameon=False, loc="upper left")

    plt.show()

In [ ]:
boxplot_vs_bits(
    [results[s]["attempts"] for s in sizes],
    title="Brute force: attempts to find the pattern",
    ylabel="Attempts (log scale)",
    log_base=2,
    yticks=[2**k for k in range(0, max(sizes) + 5, 4)],
    baseline=[2**s for s in sizes],
    baseline_label="Expected attempts $2^n$",
)

In [ ]:
boxplot_vs_bits(
    [results[s]["times"] for s in sizes],
    title="Brute force: time to find the pattern",
    ylabel="Time (s, log scale)",
)

Summary per pattern size. `mean / expected` compares the measured mean with the theoretical $2^n$, the `× prev` columns show how much attempts and time grow from one size to the next (theory: $2^{\Delta n}$, i.e. ×16 for +4 bits), and `µs / attempt` shows the cost of a single guess.

In [ ]:
rows = []

for size in sizes:
    attempts = results[size]["attempts"]
    times = results[size]["times"]

    rows.append({
        "bits": size,
        "expected (2^n)": 2**size,
        "median attempts": np.median(attempts),
        "mean attempts": attempts.mean(),
        "mean / expected": attempts.mean() / 2**size,
        "median time (s)": np.median(times),
        "mean time (s)": times.mean(),
        "µs / attempt": times.sum() / attempts.sum() * 1e6,
    })

summary = pd.DataFrame(rows).set_index("bits")
summary.insert(4, "attempts × prev", summary["mean attempts"] / summary["mean attempts"].shift())
summary.insert(7, "time × prev", summary["mean time (s)"] / summary["mean time (s)"].shift())

summary[["median attempts", "mean attempts"]] = summary[["median attempts", "mean attempts"]].round().astype(int)

pd.set_option("display.float_format", "{:.3g}".format)
summary

**Discussion**

*Attempts.* The measured means follow the theoretical $2^n$ line: `mean / expected` stays between 0.67 and 1.19. The deviations are sampling noise. For a geometric distribution the standard deviation is about the mean, so the mean of 30 trials has a standard error of $2^n/\sqrt{30} \approx 18\%$ of $2^n$. The distribution is strongly right-skewed: the median is below the mean (theory: median $\approx \ln 2 \cdot 2^n \approx 0.69 \cdot 2^n$), the upper whiskers are long and there are high outliers. Within one size the 30 trials span about two orders of magnitude, so a single run says little about how hard a size is. That is why a distribution (box-plot) is needed rather than a single number.

*Growth.* Each +4 bits multiplies the mean attempts by about $2^4 = 16$ (measured 12–22×, again noisy). Brute force is $O(2^n)$: from about 3 attempts at 2 bits to about 11 million at 24 bits. It uses no information from wrong guesses, and with replacement it can repeat a pattern already tried. That makes it about twice as expensive on average as a systematic enumeration ($\approx 2^{n-1}$).

*Time.* The cost of one attempt (generate + compare) is constant at about 0.26–0.31 µs, so time = attempts × constant. The time box-plots therefore have the same shape as the attempt box-plots and grow by the same ~16× per +4 bits. At 24 bits one trial takes 3.2 s on average (1.45 s median), and all 30 trials take about 1.5 minutes. For 2–4 bits the times are below 5 µs, where timer resolution and Python overhead dominate, so those boxes are mostly noise.

*Limit.* At about 0.29 µs per attempt, 30 trials of 28 bits need about $30 \cdot 2^{28} \cdot 0.29\,\mu s \approx 39$ minutes on average, more than the 15-minute budget, so the run stops there (see the log above). 32 bits would be another 16× (about 10 hours).

**Pattern size for the next exercises: 24 bits.** It is the largest size whose 30 trials finish within the budget, and brute force on it still runs in a reasonable time (~1.5 min) to use as a baseline.